# Preprocessing Data for PerturbGen - full adipocytes

In [2]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io
import matplotlib.pyplot as plt
import os
import anndata 
import gc
from scipy.sparse import csr_matrix

seed=42
np.random.seed(seed)


main_path = "/rds/general/user/ap5625/home/perturbation_modeling"

extra_path = "/rds/general/user/ap5625/projects/lms-scott-raw/live/T2D_REM/Phase_2/objects"

input_path = "/rds/general/user/ap5625/projects/lms-scott-raw/live/Ada/perturbation_modeling"

In [ ]:
# will not subset to only GeneFormer genes - will let tokenisation run and then 

In [3]:
adata = sc.read_h5ad(extra_path + "/Adipocytes_annotated_step2.h5ad")

In [4]:
adata.var_names_make_unique()
adata.obs_names_make_unique()  


In [5]:
adata.X = adata.layers['raw'].copy()

# Map Ensemble IDs

In [6]:
# map to ENSEMBL IDs 

ref = pd.read_csv(main_path + "/mart_export.txt", sep="\t")

print(ref.columns.tolist())


['Gene stable ID', 'Gene name']


In [7]:
gc.collect()

31

In [8]:
symbol_to_ensembl = dict(zip(ref['Gene name'], ref['Gene stable ID']))


def map_to_ensembl(gene_name):
    if str(gene_name).startswith('ENSG'):
        return gene_name
    else:
        return symbol_to_ensembl.get(gene_name, None)

adata.var['ensembl_id'] = [map_to_ensembl(g) for g in adata.var_names]


In [9]:

missing = adata.var['ensembl_id'].isna().sum()
print(f"Still unmatched: {missing}")
print(adata.var[adata.var['ensembl_id'].isna()].index.tolist()[:20])

adata = adata[:, adata.var['ensembl_id'].notna()].copy()


Still unmatched: 310
['C1orf127', 'ARHGEF19-AS1', 'NCMAP-DT', 'KIAA1522', 'ZC3H12A-DT', 'C1orf109', 'C1orf50-AS1', 'LEXM', 'LINC01135', 'LINC01358', 'HHLA3', 'HHLA3-AS1', 'LINC02797', 'ALG14-AS1', 'LINC02785', 'SLC16A4-AS1', 'PIFO', 'LINC01357', 'LINC02804', 'LINC01731']


In [10]:
gc.collect()

3270

## Ensure no dupliactes

In [11]:
# catch many-to-one symbol collisions before they become duplicate var_names
dup_mask = adata.var['ensembl_id'].duplicated(keep=False)
print(f"Rows collapsing onto a shared Ensembl ID: {dup_mask.sum()}")
print(adata.var.loc[dup_mask, ['ensembl_id']].assign(original_symbol=adata.var_names[dup_mask]).sort_values('ensembl_id'))


Rows collapsing onto a shared Ensembl ID: 18
                      ensembl_id  original_symbol
ENSG00000224516  ENSG00000224516  ENSG00000224516
ECEL1P2          ENSG00000224516          ECEL1P2
ZNF32-AS2        ENSG00000228426        ZNF32-AS2
ENSG00000228426  ENSG00000228426  ENSG00000228426
ENSG00000228999  ENSG00000228999  ENSG00000228999
LINC01830        ENSG00000228999        LINC01830
ENSG00000236924  ENSG00000236924  ENSG00000236924
LINC02851        ENSG00000236924        LINC02851
ENSG00000249388  ENSG00000249388  ENSG00000249388
FLJ12825         ENSG00000249388         FLJ12825
ENSG00000290461  ENSG00000290461  ENSG00000290461
CYP4F35P         ENSG00000290461         CYP4F35P
ENSG00000290476  ENSG00000290476  ENSG00000290476
SUGT1P3          ENSG00000290476          SUGT1P3
FAM41AY2         ENSG00000290726         FAM41AY2
ENSG00000290726  ENSG00000290726  ENSG00000290726
ENSG00000291261  ENSG00000291261  ENSG00000291261
GOLGA8IP         ENSG00000291261         GOLGA8IP


In [12]:

keep_mask = ~adata.var['ensembl_id'].duplicated(keep='first')
adata = adata[:, keep_mask].copy()
assert not adata.var['ensembl_id'].duplicated().any()


In [13]:
adata.var_names = adata.var['ensembl_id'].values
print(f"Final gene count: {adata.n_vars}")

Final gene count: 38287


# map back ensemble ids to gene symbols

In [15]:
ensembl_to_symbol = dict(zip(ref['Gene stable ID'], ref['Gene name']))

adata.var['gene_symbol'] = adata.var_names.map(ensembl_to_symbol)

# ensembl_id column is already set as var_names but add it as a column too
adata.var['ensembl_id'] = adata.var_names

# Verify
print(adata.var[['ensembl_id', 'gene_symbol']].head(10))

                      ensembl_id  gene_symbol
ENSG00000308415  ENSG00000308415      DDX11L2
ENSG00000243485  ENSG00000243485  MIR1302-2HG
ENSG00000237613  ENSG00000237613      FAM138A
ENSG00000290826  ENSG00000290826          NaN
ENSG00000186092  ENSG00000186092        OR4F5
ENSG00000238009  ENSG00000238009          NaN
ENSG00000239945  ENSG00000239945          NaN
ENSG00000239906  ENSG00000239906          NaN
ENSG00000241860  ENSG00000241860          NaN
ENSG00000241599  ENSG00000241599          NaN


In [16]:
print(f"Total genes: {adata.n_vars}")
print(f"Genes with symbol: {adata.var['gene_symbol'].notna().sum()}")


Total genes: 38287
Genes with symbol: 25798


# only keep the genes with symbols - exlcude ENS...

In [17]:
# only keep genes that also have gene symbols
has_symbol = adata.var['gene_symbol'].notna()
print(f"Total genes: {adata.n_vars}")
print(f"Genes with symbol: {has_symbol.sum()}")


Total genes: 38287
Genes with symbol: 25798


In [18]:

# subset
adata = adata[:, has_symbol].copy()
print(f"After subsetting: {adata.n_vars}")

After subsetting: 25798


In [19]:
assert adata.var['ensembl_id'].is_unique, "Duplicates reappeared in ensembl_id column"
assert adata.var_names.is_unique, "Duplicates reappeared in var_names"

In [20]:
# 5. Set var_names to ensembl_id
adata.var_names = adata.var["ensembl_id"]

In [21]:
print(f"adata now has {adata.n_vars} genes with Ensembl IDs as var_names.")

adata now has 25798 genes with Ensembl IDs as var_names.


In [22]:
adata.var.index.name = 'gene_id'

In [23]:
adata.var

,highly_variable,means,dispersions,dispersions_norm,highly_variable_nbatches,highly_variable_intersection,mean,std,ensembl_id,gene_symbol
gene_id,,,,,,,,,,
ENSG00000308415,False,2.478086e-07,0.956066,-1.696796,0,False,2.345296e-13,0.004689,ENSG00000308415,DDX11L2
ENSG00000243485,False,5.158196e-05,1.591590,-0.284362,0,False,-4.783231e-13,0.010938,ENSG00000243485,MIR1302-2HG
ENSG00000237613,False,0.000000e+00,0.000000,0.000000,0,False,0.000000e+00,1.000000,ENSG00000237613,FAM138A
ENSG00000186092,False,7.250893e-07,1.481808,-0.528350,0,False,-2.902492e-14,0.003382,ENSG00000186092,OR4F5
ENSG00000284733,False,0.000000e+00,0.000000,0.000000,0,False,0.000000e+00,1.000000,ENSG00000284733,OR4F29
...,...,...,...,...,...,...,...,...,...,...
ENSG00000198886,False,1.116456e-01,1.403198,-0.978742,0,False,1.641235e-11,0.380059,ENSG00000198886,MT-ND4
ENSG00000198786,False,5.438835e-02,1.527072,-0.654929,0,False,1.288830e-11,0.225062,ENSG00000198786,MT-ND5
ENSG00000198695,False,3.082701e-02,1.244590,-1.832841,0,False,6.944349e-12,0.099782,ENSG00000198695,MT-ND6


# obs pre-processing

In [53]:
adata_sub = adata[adata.obs["condition"].isin(["baseline", "weightloss"])].copy()

In [54]:
gc.collect()

1974

In [55]:
print(adata.obs['condition'].value_counts())
print(adata_sub.obs['condition'].value_counts())


condition
weightloss    91273
baseline      56955
Lean           7695
Name: count, dtype: int64
condition
weightloss    91273
baseline      56955
Name: count, dtype: int64


In [56]:
adata_sub.obs['donor_id'] = adata_sub.obs['Donor']

In [57]:
adata_sub.obs['condition'] = adata_sub.obs['condition'].map({
    'baseline': 'obese',
    'weightloss': 'weightloss'
})

In [59]:
print(adata_sub.obs['condition'].value_counts())


condition
weightloss    91273
obese         56955
Name: count, dtype: int64


In [60]:
# keep only explicitly paired

obese_donors = set(adata_sub.obs[adata_sub.obs['condition'] == 'obese']['Donor'].unique())
weight_loss_donors = set(adata_sub.obs[adata_sub.obs['condition'] == 'weightloss']['Donor'].unique())


In [61]:

paired = baseline_donors & weight_loss_donors

len(paired)



66

In [62]:

adata_sub = adata_sub[adata_sub.obs['Donor'].isin(paired)].copy()


In [63]:
gc.collect()

3336

In [64]:
adata_sub.obs['cell_states_adipocytes'] = adata_sub.obs['cell_state_t2d']

## check duplicates

In [65]:
dup_mask = adata_sub.var_names.duplicated(keep=False)
print(f"Total rows involved in duplicates: {dup_mask.sum()} / {adata_sub.n_vars}")
print(adata_sub.var_names[dup_mask].value_counts().head(20))

Total rows involved in duplicates: 0 / 25798
Series([], Name: count, dtype: int64)


In [51]:
# checknthat its raw counts 
print("X max:", adata_sub.X.max(), " — should look like raw integer counts, not small floats")
print("X min:", adata_sub.X.min(), " — should be >= 0")


X max: 10267.0  — should look like raw integer counts, not small floats
X min: 0.0  — should be >= 0


In [66]:
for col in ['condition', 'Donor', 'cell_states_adipocytes']:
    n_missing = adata_sub.obs[col].isna().sum()
    print(f"{col}: {n_missing} missing, values = {adata_sub.obs[col].unique()[:10]}")


condition: 0 missing, values = ['obese', 'weightloss']
Categories (2, object): ['obese', 'weightloss']
Donor: 0 missing, values = ['DR006', 'DR001', 'DR005', 'DR012', 'DR016', 'DR015', 'MMT22', 'MMT23', 'MMT15', 'MMT08']
Categories (66, object): ['DR001', 'DR004', 'DR005', 'DR006', ..., 'P140B', 'P155B', 'P156B', 'P157B']
cell_states_adipocytes: 0 missing, values = ['AD1', 'AD2', 'AD3', 'AD8', 'AD10', 'AD6', 'AD5_BMPERhi', 'AD9', 'AD5_basal', 'AD4']
Categories (12, object): ['AD1', 'AD2', 'AD3', 'AD4', ..., 'AD8', 'AD9', 'AD10', 'Unassigned']


In [67]:
obese_donors = set(adata_sub.obs.loc[adata_sub.obs['condition'] == 'obese', 'Donor'].unique())
wl_donors = set(adata_sub.obs.loc[adata_sub.obs['condition'] == 'weightloss', 'Donor'].unique())
print(f"Obese donors: {len(obese_donors)}, WL donors: {len(wl_donors)}")
print(f"Paired (in both): {len(obese_donors & wl_donors)}")
print(f"Unpaired donors present: {len(obese_donors ^ wl_donors)}")

Obese donors: 66, WL donors: 66
Paired (in both): 66
Unpaired donors present: 0


In [68]:
adata_sub.write(input_path + "/adipocytes_full_perturbgen_preprocessed_paired.h5ad")

In [25]:
print(adata_sub)

AnnData object with n_obs × n_vars = 148228 × 25807
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'doublet_score', 'predicted_doublet', 'barcodes', 'condition', 'remission', 't2d_batch', 'n_genes', 'count_keep', 'mt_keep', 'rb_keep', 'doublet_keep', 'study', 'pooled_donors', 'SeqID', 'cellID', 'Donor', 'sample', 'Chemistry', 'pool_barcodes', 'dataset', 'BMI', 'sex', 'mt.percent', 'ribo.percent', 'emont_cell_type', 'emont_cell_type2', 'cell_type_level1', 'cell_type_level2', 'cell_state', 'subUMAP1_v1', 'subUMAP2_v1', 'subUMAP1_v2', 'subUMAP2_v2', 'New_donor', 'new_sample', 'UMAPglobal_x', 'UMAPglobal_y', 'orig_barcodes', 'miranda_barcodes', 'leiden_0.1', 'leiden_0.15', 'leiden_0.25', 'leiden_